[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/03_Training_Strategies/03_training_pipeline.ipynb)

# 03. Full Training Pipeline from Scratch

**This is the most important notebook for learning HOW TO TRAIN.**

**This notebook covers:**
- Complete training loop with all best practices
- Data pipeline (loading, augmentation, batching)
- Optimizer & scheduler choices for multimodal
- Gradient accumulation (simulate large batches on small GPU)
- Mixed precision training (2x speed, half memory)
- Checkpointing and resuming
- Full training run with live visualization

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/03_Training_Strategies")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
import numpy as np
import time
import os
from tqdm import tqdm
from utils.visualization import *
from utils.helpers import *

set_style()
device = get_device()

## 1. Training Recipe Overview

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Multimodal Training Pipeline', fontsize=18, fontweight='bold', pad=20)

steps = [
    (7, 9.2, 'Data Pipeline\n(images + texts + augmentation)', '#E74C3C'),
    (7, 7.8, 'Forward Pass\n(encode image → encode text → compute loss)', '#3498DB'),
    (7, 6.4, 'Backward Pass\n(compute gradients, gradient accumulation)', '#F39C12'),
    (7, 5.0, 'Optimizer Step\n(AdamW + gradient clipping)', '#2ECC71'),
    (7, 3.6, 'Scheduler Step\n(cosine decay with warmup)', '#9B59B6'),
    (7, 2.2, 'Logging & Checkpointing\n(loss, metrics, save model)', '#1ABC9C'),
    (7, 0.8, 'Evaluation\n(validation loss, retrieval accuracy)', '#34495E'),
]

for i, (x, y, label, color) in enumerate(steps):
    draw_architecture_block(ax, x, y, 9, 0.8, label, color, fontsize=10)
    if i < len(steps) - 1:
        draw_arrow(ax, (x, y - 0.5), (x, steps[i+1][1] + 0.5))

# Loop arrow
ax.annotate('', xy=(12, 9.2), xytext=(12, 0.8),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2, ls='--',
                           connectionstyle='arc3,rad=0.3'))
ax.text(13, 5, 'Repeat\nfor N\nepochs', fontsize=10, color='gray', ha='center')

plt.tight_layout()
plt.savefig('../assets/training_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Data Pipeline

In [ ]:
# Create synthetic data (no download needed)
images, texts, labels = create_synthetic_image_text_pairs(
    n_samples=500, img_size=32, n_classes=5
)

# Simple tokenizer
all_words = set()
for t in texts:
    all_words.update(t.lower().split())
word2id = {w: i+2 for i, w in enumerate(sorted(all_words))}
word2id['[PAD]'] = 0
word2id['[CLS]'] = 1
vocab_size = len(word2id)

def tokenize(text, max_len=16):
    ids = [word2id['[CLS]']] + [word2id.get(w, 0) for w in text.lower().split()]
    ids = ids[:max_len]
    ids += [0] * (max_len - len(ids))
    return torch.tensor(ids)

# Data augmentation for images
import torchvision.transforms as T

train_transform = T.Compose([
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.RandomErasing(p=0.1),
])

class MultimodalDataset(Dataset):
    def __init__(self, images, texts, transform=None):
        self.images = images
        self.texts = texts
        self.transform = transform
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img)
        txt = tokenize(self.texts[idx])
        return img, txt

# Train/val split
n_train = int(0.8 * len(images))
train_dataset = MultimodalDataset(images[:n_train], texts[:n_train], train_transform)
val_dataset = MultimodalDataset(images[n_train:], texts[n_train:])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} samples, {len(val_loader)} batches")
print(f"Vocab: {vocab_size} words")

## 3. Model + Optimizer + Scheduler

In [ ]:
# Reuse CLIP model from Module 02

class CLIPModel(nn.Module):
    def __init__(self, embed_dim=128, proj_dim=64, vocab_size=100):
        super().__init__()
        # Image encoder
        self.img_patch = nn.Conv2d(3, embed_dim, 4, 4)
        self.img_cls = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.img_pos = nn.Parameter(torch.randn(1, 65, embed_dim) * 0.02)
        img_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=256, batch_first=True)
        self.img_transformer = nn.TransformerEncoder(img_layer, num_layers=3)
        self.img_norm = nn.LayerNorm(embed_dim)
        self.img_proj = nn.Linear(embed_dim, proj_dim)

        # Text encoder
        self.tok_embed = nn.Embedding(vocab_size, embed_dim)
        self.txt_pos = nn.Embedding(64, embed_dim)
        txt_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=256, batch_first=True)
        self.txt_transformer = nn.TransformerEncoder(txt_layer, num_layers=3)
        self.txt_norm = nn.LayerNorm(embed_dim)
        self.txt_proj = nn.Linear(embed_dim, proj_dim)

        self.logit_scale = nn.Parameter(torch.ones(1) * np.log(1 / 0.07))

    def encode_image(self, x):
        B = x.shape[0]
        x = self.img_patch(x).flatten(2).transpose(1, 2)
        x = torch.cat([self.img_cls.expand(B, -1, -1), x], dim=1)
        x = x + self.img_pos
        x = self.img_transformer(x)
        x = self.img_norm(x[:, 0])
        return F.normalize(self.img_proj(x), dim=-1)

    def encode_text(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        x = self.tok_embed(x) + self.txt_pos(pos)
        x = self.txt_transformer(x)
        x = self.txt_norm(x[:, 0])
        return F.normalize(self.txt_proj(x), dim=-1)

    def forward(self, images, text_ids):
        img_emb = self.encode_image(images)
        txt_emb = self.encode_text(text_ids)
        logit_scale = self.logit_scale.exp()
        return logit_scale * img_emb @ txt_emb.T


model = CLIPModel(embed_dim=128, proj_dim=64, vocab_size=vocab_size).to(device)
count_parameters(model)

In [ ]:
# Optimizer: AdamW (always use this for transformers)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

# Scheduler: Cosine decay with linear warmup
total_steps = len(train_loader) * 50  # 50 epochs
warmup_steps = total_steps // 10      # 10% warmup

def get_lr(step):
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, get_lr)

# Visualize learning rate schedule
lrs = [get_lr(s) * 3e-4 for s in range(total_steps)]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lrs, color='#E74C3C', linewidth=2)
ax.axvline(x=warmup_steps, color='gray', linestyle='--', label='End of warmup')
ax.set_xlabel('Training Step')
ax.set_ylabel('Learning Rate')
ax.set_title('Cosine Decay with Linear Warmup', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Total steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"Peak LR: 3e-4")

## 4. Gradient Accumulation (Essential for Low Compute!)

**Problem:** CLIP needs large batch sizes (1000+) but you only have memory for 32.  
**Solution:** Accumulate gradients over multiple mini-batches before stepping.

In [ ]:
# Visualize gradient accumulation
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Without accumulation
ax = axes[0]
ax.set_title('Without Gradient Accumulation\n(batch=32, effective=32)', fontsize=12, fontweight='bold')
steps_labels = ['Batch 1\n(32)', 'Step', 'Batch 2\n(32)', 'Step', 'Batch 3\n(32)', 'Step']
colors = ['#3498DB', '#E74C3C', '#3498DB', '#E74C3C', '#3498DB', '#E74C3C']
ax.barh(range(6), [1]*6, color=colors, alpha=0.7)
for i, label in enumerate(steps_labels):
    ax.text(0.5, i, label, ha='center', va='center', fontsize=9, fontweight='bold')
ax.set_xlim(0, 1)
ax.axis('off')

# With accumulation (4 steps)
ax = axes[1]
ax.set_title('With Gradient Accumulation (4 steps)\n(batch=32, effective=128)', 
             fontsize=12, fontweight='bold', color='#2ECC71')
steps_labels = ['Batch 1 (32)', 'Batch 2 (32)', 'Batch 3 (32)', 'Batch 4 (32)', 
                'STEP\n(accumulated)', 'Batch 5...']
colors = ['#3498DB', '#3498DB', '#3498DB', '#3498DB', '#E74C3C', '#3498DB']
bars = ax.barh(range(6), [1]*6, color=colors, alpha=0.7)
for i, label in enumerate(steps_labels):
    ax.text(0.5, i, label, ha='center', va='center', fontsize=9, fontweight='bold')
ax.set_xlim(0, 1)
ax.axis('off')

plt.tight_layout()
plt.savefig('../assets/gradient_accumulation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Full training loop with all best practices

def train_multimodal(
    model, train_loader, val_loader, optimizer, scheduler,
    n_epochs=50, grad_accum_steps=4, max_grad_norm=1.0,
    device='cpu'
):
    """Production-ready training loop for multimodal models."""
    history = {'train_loss': [], 'val_loss': [], 'lr': [], 'epoch_time': []}
    best_val_loss = float('inf')
    global_step = 0

    for epoch in range(n_epochs):
        start_time = time.time()
        model.train()
        epoch_loss = 0
        n_batches = 0

        for batch_idx, (images, text_ids) in enumerate(train_loader):
            images = images.to(device)
            text_ids = text_ids.to(device)

            # Forward pass
            logits = model(images, text_ids)
            labels = torch.arange(len(images), device=device)
            loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

            # Scale loss for gradient accumulation
            loss = loss / grad_accum_steps
            loss.backward()

            if (batch_idx + 1) % grad_accum_steps == 0:
                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                
                # Optimizer step
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

            epoch_loss += loss.item() * grad_accum_steps
            n_batches += 1

        avg_train_loss = epoch_loss / n_batches
        history['train_loss'].append(avg_train_loss)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        # Validation
        model.eval()
        val_loss = 0
        n_val = 0
        with torch.no_grad():
            for images, text_ids in val_loader:
                images = images.to(device)
                text_ids = text_ids.to(device)
                logits = model(images, text_ids)
                labels = torch.arange(len(images), device=device)
                loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2
                val_loss += loss.item()
                n_val += 1

        avg_val_loss = val_loss / max(n_val, 1)
        history['val_loss'].append(avg_val_loss)

        elapsed = time.time() - start_time
        history['epoch_time'].append(elapsed)

        # Checkpointing (save best model)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': best_val_loss,
            }, '../assets/best_model.pt')

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{n_epochs} | "
                  f"Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | "
                  f"LR: {optimizer.param_groups[0]['lr']:.6f} | Time: {elapsed:.1f}s")

    return history


# Reset model and optimizer
model = CLIPModel(embed_dim=128, proj_dim=64, vocab_size=vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
total_steps = len(train_loader) * 50 // 4  # account for grad accumulation
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

# Train!
print("Training with gradient accumulation (effective batch = 32 × 4 = 128)...")
history = train_multimodal(
    model, train_loader, val_loader, optimizer, scheduler,
    n_epochs=50, grad_accum_steps=4, device=device
)

In [ ]:
# Visualize training results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training Results', fontsize=16, fontweight='bold')

# Loss curves
ax = axes[0]
ax.plot(history['train_loss'], label='Train', color='#E74C3C', linewidth=2)
ax.plot(history['val_loss'], label='Validation', color='#3498DB', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Loss Curves')
ax.legend()

# Learning rate
ax = axes[1]
ax.plot(history['lr'], color='#2ECC71', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule')

# Epoch time
ax = axes[2]
ax.bar(range(len(history['epoch_time'])), history['epoch_time'], color='#F39C12', alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_ylabel('Time (seconds)')
ax.set_title('Epoch Duration')

plt.tight_layout()
plt.savefig('../assets/training_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTotal training time: {sum(history['epoch_time']):.1f}s")
print(f"Best validation loss: {min(history['val_loss']):.4f}")

## Training Checklist for Low Compute

| Technique | Memory Saving | Speed | Difficulty |
|-----------|--------------|-------|------------|
| **Gradient Accumulation** | None (same mem) | Same | Easy |
| **Mixed Precision (fp16)** | ~50% | ~2x | Easy |
| **Gradient Checkpointing** | ~60% | ~0.8x | Medium |
| **Freeze Encoders** | ~50%+ | ~2x | Easy |
| **LoRA/QLoRA** | ~75-90% | ~1.5x | Medium |
| **Smaller Model** | Proportional | Proportional | Easy |

---
**Next:** Module 04 - Finetuning for Low Compute (LoRA, QLoRA, Adapters!)